# CounselRAG: A Contextually-Aware Legal QA Platform

### Introduction

CounselRAG is a legal question-answering platform that combines Retrieval-Augmented Generation (RAG) with Knowledge Graphs to deliver accurate, context-rich responses to legislative queries. The platform explores how smaller, fine-tuned LLMs can be made more capable by integrating embeddings from user-provided legal documents (persisted in a FAISS vector store) and structured knowledge from a Neo4j-powered graph built from curated Wikipedia articles on U.S. and international law.

### Our Proposed Architecture

![Architecture](images/architecture.png)

### Persisting the Legal Document into FAISS

In [3]:
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.vectorstores.faiss import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

Environment Variables

In [ ]:
FAISS_INDEX_PATH = os.path.join(os.getcwd(), "faiss_index")
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "XXXX"
GROQ_API_KEY = "XXXX"
GROQ_MODEL = "llama3-8b-8192"
GROQ_ENDPOINT = "https://api.groq.com/openai/v1/chat/completions"

* We upload the given legal PDF documents and text files into a particular folder. We iterate through the files of the folder and create chunks to store as embeddings into the vector storage index (FAISS)
* We split the document into overlapping documents i.e chunks of ~1000 characters, overlapping by 250 characters, to preserve context.

In [5]:
def get_text_chunks_langchain(folder_name):
    source_chunks = []
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
    for root, _, files in os.walk(folder_name):
        for file in files:
            file_path = os.path.join(root, file)
            if file.endswith(".txt"):
                loader = TextLoader(file_path)
            elif file.endswith(".pdf"):
                loader = PyPDFLoader(file_path)
            docs = loader.load()
            for doc in docs:
                doc.metadata["source"] = os.path.basename(file_path)
            chunks = splitter.split_documents(docs)
            source_chunks.extend(chunks)
    print(f"[INFO] Total chunks created: {len(source_chunks)}")
    return source_chunks

Process the generated chunks by converting them into their respective embedding form, which is then persisted into the DB. This enables similarity searching and retreival later in our RAG pipeline

In [ ]:
def process_chunks(chunks):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db = FAISS.from_documents(chunks, embeddings)
    return db

In [7]:
def persist_data(folder_name):
    chunks = get_text_chunks_langchain(folder_name)
    db = process_chunks(chunks)
    db.save_local(FAISS_INDEX_PATH)

### Retreive additional context from the Knowledge Graph

In [ ]:
from neo4j import GraphDatabase

In [36]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

This is the main function - get_relevant_nodes() where we perform a vector similarity search on the Neo4j graph using an embedding input of the PDF Context. It returns the top K most relevant nodes with their metadata and similarity scores!

In [38]:
def get_relevant_nodes(embedded_query, top_k):
    cypher_query = f"""
    CALL db.index.vector.queryNodes('relevant_entity_embeddings_index', $k, $embedding)
    YIELD node, score
    RETURN elementId(node) AS node_id, node.name AS name, node.description AS description, labels(node) AS labels, score
    ORDER BY score DESC
    """
    with driver.session() as session:
        results = session.run(cypher_query, embedding=embedded_query, k=top_k)
        return [{"id": r["node_id"], "name": r["name"], "description": r.get("description", ""), "labels": r["labels"], "score": r["score"]} for r in results]

In `expand_neighbors` we take a list of node IDs and retrieve all their immediate neighbors (left and right) from Neo4j. It's then formatted into clear, readable strings showing the two nodes and their relationship, with the node descriptions.

A future improvement would be to increase the traversal of nodes' neighbours.

In [31]:
def expand_neighbors(node_ids):
    cypher_query = """
    MATCH (n)-[r]-(m)
    WHERE elementId(n) IN $node_ids
    RETURN n.name AS left_node, n.description AS left_desc, type(r) AS relationship, m.name AS right_node, m.description AS right_desc
    LIMIT 100
    """
    with driver.session() as session:
        results = session.run(cypher_query, node_ids=node_ids)
        neighbour_nodes = []
        for r in results:
            left_desc = r["left_desc"]
            right_desc = r["right_desc"]
            neighbour_nodes.append(f"{r['left_node']} ({left_desc}) --[{r['relationship']}]-- {r['right_node']} ({right_desc})")
        return neighbour_nodes

The below function takes the context of the PDF document, performs a similarity search on the knowledge graph, retrieves relevant nodes and their neighbors, and builds a structured text summary i.e context block that is used as input in the created prompt for the LLM.

In [ ]:
def get_full_context(pdf_context):
    embedded_query = embedding_model.embed_query(pdf_context)
    top_matching_nodes = get_relevant_nodes(embedded_query, top_k = 5)
    
    if not top_matching_nodes:
        print("No relevant nodes found.")
        return
    
    print("Top Retrieved Nodes:")
    for node in top_matching_nodes:
        print(f"{node['name']} (Similarity Score: {node['score']})")
        print(f"Description: {node.get('description', '')}")

    node_ids = [node['id'] for node in top_matching_nodes]
    neighbor_information = expand_neighbors(node_ids)
    
    graph_contexts = []
    
    for node in top_matching_nodes:
        description = node.get('description', '')
        graph_contexts.append(f"{node['name']} (Labels: {', '.join(node['labels'])}) {description}")
    
    if neighbor_information:
        for neighbour in neighbor_information:
            graph_contexts.append(neighbour)
    
    context_block = "\n".join(graph_contexts)
    
    return context_block

## Final RAG + Knowledge Graph QA Pipeline

#### Prompt Template

In [ ]:
import os
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import pipeline
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores.faiss import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# from context_from_kg import get_full_context
import requests
from langchain_core.runnables import RunnableLambda
from langchain_ollama import ChatOllama

In [ ]:
PROMPT_TEMPLATE = '''
You are a legal domain expert assistant. Given the following excerpts from legal documents and case texts, answer the question using only the provided context.

•⁠  If the question is simple or asks for a fact or definition, give a concise, direct answer (one or two sentences).
•⁠  If the question is broad, complex, or asks for reasoning, provide a detailed answer (about 5-6 sentences) including relevant legal reasoning and case details.
•⁠  Avoid making up any information not present in the context.
•⁠  For complex questions, first identify which retrieved excerpts are most relevant, then synthesize them into a coherent legal explanation.

---
PDF Retrieved Context:
{pdf_context}

---
Knowledge Graph Context:
{kg_context}

---
Question:
{question}

---
Answer:
'''

We built a prompt with the query from the user, context from the uploaded PDF files and external knowledge from the knowledge graph!

In [19]:
def build_prompt(pdf_context, kg_context, question):
    return PROMPT_TEMPLATE.format(
        pdf_context=pdf_context,
        kg_context=kg_context,
        question=question
    )

The `call_groq_llm()` function sends a system prompt and user prompt to the Groq LLM API, retrieves the generated response, and returns the text output. This function will make it easier for us to pass our own user prompt created above

In [21]:
def call_groq_llm(system_prompt: str, user_prompt: str):
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": GROQ_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.7
    }
    response = requests.post(GROQ_ENDPOINT, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content']

The below function loads the FAISS vector index, and retreives the top 4 most similar content, compared with the user query from the db. This PDF context is utilized in the user prompt

In [ ]:
def get_pdf_context(query):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db = FAISS.load_local(FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
    retriever = db.as_retriever(search_kwargs={"k": 4}, search_type="mmr")
    docs = retriever.get_relevant_documents(query)
    if not docs:
        return "No relevant documents found"
    return "\n\n".join(doc.page_content for doc in docs)

* The prompt() function orchestrates the end-to-end legal question-answering workflow by combining document retrieval, knowledge graph context, and large language model (LLM) inference

* We first retrieve relevant PDF and knowledge graph contexts for the input question. 

* Depending on the selected LLM backend, we dynamically route the prompt. If the llama3-8b-8192 model is selected, it constructs a combined prompt and sends it to the Groq API via the call_groq_llm() function. For models like gemma:2b, granite3.3:2b, or gemma3:4b, we set up a RAG chain using Ollama.

* This ensures the system can flexibly switch between different LLM backends while maintaining a consistent pipeline for answering high-quality legal questions.

In [ ]:
def prompt(question, model):
    pdf_context = get_pdf_context(question)
    kg_context = get_full_context(pdf_context)

    if model == "llama3-8b-8192":
        user_prompt = build_prompt(pdf_context, kg_context, question)
        answer = call_groq_llm(
            system_prompt="You are a expert legal QA assistant",
            user_prompt=user_prompt
        )
        return answer

    elif model in ["gemma:2b", "granite3.3:2b", "gemma3:4b"]:
        llm = ChatOllama(
            model=model,
            temperature=0.7,
        )
        
        qaprompt = PromptTemplate.from_template(PROMPT_TEMPLATE)
        kg_context_runnable = RunnableLambda(lambda input: get_full_context(input["pdf_context"]))

        rag_chain = (
            {
                "pdf_context": RunnablePassthrough().bind(value=pdf_context),
                "kg_context": kg_context_runnable,
                "question": RunnablePassthrough(),
            }
            | qaprompt
            | llm
            | StrOutputParser()
        )

        return rag_chain.invoke({"question": question, "pdf_context": pdf_context})
    else:
        raise ValueError(f"Unsupported model: {model}")

In [40]:
prompt("What is the case about?", "gemma:2b")

Top Retrieved Nodes:
Skadden, Arps, Slate, Meagher & Flom LLP and Affiliates (Similarity Score: 0.6951785087585449)
Description: None
American Civil Liberties Union (Similarity Score: 0.6529281139373779)
Description: None
University of Chicago Law School (Similarity Score: 0.6473000049591064)
Description: None
Stephen G. Breyer (Similarity Score: 0.6383023262023926)
Description: None
Columbia Law School (Similarity Score: 0.6351044178009033)
Description: None
Top Retrieved Nodes:
Skadden, Arps, Slate, Meagher & Flom LLP and Affiliates (Similarity Score: 0.6951785087585449)
Description: None
American Civil Liberties Union (Similarity Score: 0.6529281139373779)
Description: None
University of Chicago Law School (Similarity Score: 0.6473000049591064)
Description: None
Stephen G. Breyer (Similarity Score: 0.6383023262023926)
Description: None
Columbia Law School (Similarity Score: 0.6351044178009033)
Description: None


"The passage discusses the Board's interpretation of the NLRA and its application to employee cases. The passage argues that the Board's interpretation of the NLRA is not deferrable to other and equally important congressional objectives. The passage also cites a case where the Board's interpretation of a statute was reversed."

In [41]:
prompt("What remedy did the NLRB propose?", "gemma3:4b")

Top Retrieved Nodes:
Skadden, Arps, Slate, Meagher & Flom LLP and Affiliates (Similarity Score: 0.6951785087585449)
Description: None
American Civil Liberties Union (Similarity Score: 0.6529281139373779)
Description: None
University of Chicago Law School (Similarity Score: 0.6473000049591064)
Description: None
Stephen G. Breyer (Similarity Score: 0.6383023262023926)
Description: None
Columbia Law School (Similarity Score: 0.6351044178009033)
Description: None
Top Retrieved Nodes:
Skadden, Arps, Slate, Meagher & Flom LLP and Affiliates (Similarity Score: 0.6951785087585449)
Description: None
American Civil Liberties Union (Similarity Score: 0.6529281139373779)
Description: None
University of Chicago Law School (Similarity Score: 0.6473000049591064)
Description: None
Stephen G. Breyer (Similarity Score: 0.6383023262023926)
Description: None
Columbia Law School (Similarity Score: 0.6351044178009033)
Description: None


'The NLRB proposed the following remedies: (1) cease and desist from further violations of the NLRA, (2) post a detailed notice to its employees regarding the remedial order, and (3) offer reinstatement and backpay to the union supporters.'

### Manual Evaluation

<img src="images/manual_evaluations.png" alt="Manual Evaluation" style="border: 2px solid black; border-radius: 4px;">

## References

- FAISS : https://github.com/facebookresearch/faiss
- https://www.anyscale.com/blog/turbocharge-langchain-now-guide-to-20x-faster-embedding
- RAG : https://python.langchain.com/docs/tutorials/rag/
- RAG 2 : https://python.langchain.com/docs/tutorials/qa_chat_history/
- PyPDFLoader : https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/
- Sentence Transformer : https://python.langchain.com/docs/integrations/text_embedding/sentence_transformers/
- Loading a PDF with Langchain : https://python.langchain.com/docs/how_to/document_loader_pdf/
- GraphDatabase : https://neo4j.com/docs/api/python-driver/current/api.html
- Text Embedding : https://python.langchain.com/docs/how_to/embed_text/
- Vector Index : https://neo4j.com/docs/cypher-manual/current/indexes/semantic-indexes/vector-indexes/
- Traversal of Nodes : https://neo4j.com/docs/cypher-manual/current/clauses/match/
- Similarity Search : https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/
- Query Embedding : https://api.python.langchain.com/en/latest/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings.html
- Prompt Template : https://medium.com/@ssmaameri/prompt-templates-in-langchain-efb4da260bd3
- https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html
- Groq - https://github.com/groq/groq-python
- https://blog.donvitocodes.com/integrating-generative-ai-with-real-time-data-from-apis-groq-python-and-go
- ChatOllama Integration : https://python.langchain.com/docs/concepts/chat_models/
- https://python.langchain.com/docs/integrations/chat/ollama/
- https://medium.com/data-science-collective/langchain-mcp-rag-ollama-the-key-to-powerful-agentic-ai-91529b2fa320
- https://medium.com/towards-agi/how-to-use-ollama-effectively-with-langchain-tutorial-546f5dbffb70
- Runnables : https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.passthrough.RunnablePassthrough.html
- Streamlit : https://docs.streamlit.io/develop/api-reference/widgets/st.file_uploader
- Streamlit 2: https://docs.streamlit.io/develop/concepts/architecture/session-state
- Streamlit 3: https://docs.streamlit.io/develop/api-reference/chat 

### Contribution Table

| Team Member | Project Part# | Contribution (%) |
|---|---|---|
| Aayush Subramaniam, Meghna Verma | Knowledge Graph | 50-50  |
| Aayush Subramaniam, Meghna Verma | FAISS Vector Store | 50-50   |
| Aayush Subramaniam, Meghna Verma | Prompt Builing & RAG Chain | 50-50   |
| Aayush Subramaniam, Meghna Verma | Streamlit UI | 50-50   |
| Aayush Subramaniam, Meghna Verma | **Total** | 50-50  |